# Probe: xml.etree.ElementTree
Each cell states how one method behaves, as `assert`s. If a claim is wrong, the cell fails.

In [1]:
from xml.etree.ElementTree import fromstring

doc = fromstring("""
<form>
    <issuer>
        <cik>0000097476</cik>
        <ticker></ticker>
        <name>  TEXAS INSTRUMENTS  </name>
    </issuer>
    <owner><cik>1</cik></owner>
    <owner><cik>2</cik></owner>
    <line><price><value>258.14</value><footnoteId id="F1"/></price></line>
</form>
""")

## An Element is: tag, text, attrib, children

In [2]:
issuer = doc.find("issuer")
assert issuer.tag == "issuer"
assert issuer.text.strip() == ""                        # a container's text is only whitespace
assert [c.tag for c in issuer] == ["cik", "ticker", "name"]  # children, in document order
assert doc.find("line/price/footnoteId").attrib == {"id": "F1"}

## `findtext`: the text at a path

In [3]:
assert doc.findtext("issuer/cik") == "0000097476"       # always a str: leading zeros kept, no number conversion
assert doc.findtext("issuer/ticker") == ""              # present but empty  → ""
assert doc.findtext("issuer/missing") is None           # absent             → None
assert doc.findtext("issuer/missing", "?") == "?"       # absent, with a default
assert doc.findtext("issuer/name") == "  TEXAS INSTRUMENTS  "  # whitespace is NOT stripped

## `find`: one element (the first match)

In [4]:
assert doc.find("owner").findtext("cik") == "1"          # first match only
assert doc.find("missing") is None                      # absent → None (so .findtext on it would crash)

## `findall`: every match, as a list

In [5]:
owners = doc.findall("owner")
assert [o.findtext("cik") for o in owners] == ["1", "2"]  # all matches, in document order
assert doc.findall("missing") == []                     # absent → empty list, never None

## Paths are relative to the element you call on

In [6]:
assert doc.findtext("cik") is None                      # cik is not a DIRECT child of form
assert issuer.findtext("cik") == "0000097476"           # but it is a child of issuer
assert doc.findtext(".//cik") == "0000097476"           # .// searches at any depth (first match)

## The `<value>` wrapper: the data is one level down

In [7]:
assert doc.findtext("line/price").strip() == ""         # the wrapper itself holds no data
assert doc.findtext("line/price/value") == "258.14"     # the value is inside <value>

## Consequences for parsing
- `findtext` gives `None` (absent), `""` (empty) or a string with whitespace: normalise all three, so missing has **one** meaning.
- Every value is a `str`: converting to numbers and dates is our job, and can fail.
- Use `findall` for anything that repeats (owners, lines), since it gives `[]` rather than `None`.
- Footnotable fields keep their data in `<value>`.